# Phase 1 — Meer: Corpus QA, Annotation Pilot, Validation Protocol

**RBI-ObliBench / Agentic RAG compliance system**

Runs an independent QA pass over Akash's corpus, the two Week-2 risk checks (cross-class alignment against the 60% trigger, and the FAQ/enforcement source check), and generates the annotation pilot. Thin wrapper only — every cell calls into `src/` or `scripts/run_annotation.py`.

**Add Input** the `rbi-corpus-v1` and `rbi-matrix-v1` Datasets (from P1-001 and P1-002) before running.

> This notebook **cannot** produce a Fleiss' kappa on its own. It generates task files; three human annotators fill them in; `ingest` then computes agreement. No annotations are fabricated at any point.

## 1. Get the code

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/karanLokhande29/Capstone_project.git"
BRANCH = "main"

WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
REPO_DIR = os.path.join(WORKING, "Capstone_project")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # /kaggle/working survives across "Run All" in one session — always sync
    # rather than trusting a stale checkout (cost three debug rounds in P1-001).
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True,
).stdout.strip()
print("repository:", REPO_DIR)
print("commit:     ", commit, "-- check this matches the latest commit on GitHub")

## 2. Attach the corpus and matrix

Copies any attached Dataset's `data/` into the writable working root — `/kaggle/input` is read-only, so the pipeline can never write there.

In [ ]:
import shutil
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DATA = Path(REPO_DIR) / "data"

if KAGGLE_INPUT.is_dir():
    for dataset in sorted(d for d in KAGGLE_INPUT.iterdir() if (d / "data").is_dir()):
        source = dataset / "data"
        print(f"copying from {source}")
        for sub in ("metadata", "processed", "extracted", "matrix"):
            src, dst = source / sub, WORKING_DATA / sub
            if src.is_dir():
                dst.mkdir(parents=True, exist_ok=True)
                for f in src.iterdir():
                    if f.is_file():
                        shutil.copy2(f, dst / f.name)
else:
    print("not on Kaggle — using the repo's own data/ as-is")

processed = sorted((WORKING_DATA / "processed").glob("md_*.jsonl"))
print(f"processed documents available: {len(processed)}")

## 3. Tests

Protocol enforcement (tautology guard, promotion gates, kappa edge cases) plus the end-to-end pilot integration test against real committed paragraphs.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_benchmark_annotation.py",
     "tests/test_benchmark_integration.py", "-v"],
    capture_output=True, text=True,
)
print(result.stdout[-6000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
print("TESTS:", "PASS" if result.returncode == 0 else "FAIL")

## 4. Corpus QA, Week-2 risk checks, and pilot generation

Runs all three stages and writes `reports/phase1_meer_annotation.md`. The cross-class alignment result is a **decision point**, not just a number — read the judgment line in the output.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/run_annotation.py", "all", "--json"],
    capture_output=True, text=True,
)
print(result.stdout[-8000:])
if result.returncode != 0:
    print(result.stderr[-4000:])

## 5. The 60% alignment trigger — read this output

Printed on its own because it is the one specified threshold in this prompt and it changes what Phase 2 should build.

In [ ]:
import json
from src.common.config import load_config
from src.common.paths import PathResolver
from src.benchmark.alignment_check import cross_class_alignment

cfg = load_config()
resolver = PathResolver.from_config(cfg)
alignment = cross_class_alignment(cfg, resolver=resolver)

ent = alignment["entity_class_axis"]
base = alignment["subject_family_axis_derived"]
print(f"paragraph-level parallel : {ent['paragraph_level']['alignment_rate']}")
print(f"paragraph-level baseline : {base['paragraph_level']['alignment_rate']}")
print(f"section-level   parallel : {ent['section_level']['alignment_rate']}")
print(f"TRIGGER FIRED            : {alignment['trigger_fired']}")
print()
print(alignment["judgment"])

## 6. Annotation task files

Three files, one per annotator, each carrying every pilot item (full overlap is what makes an agreement statistic computable at this scale). Download these, have Akash / Karan / Meer fill them in per `data/benchmark/templates/ANNOTATION_INSTRUCTIONS.md`, then run the ingest cell.

In [ ]:
from pathlib import Path

tasks = sorted(Path("data/benchmark/tasks").glob("annotation_*.csv"))
for path in tasks:
    rows = sum(1 for _ in open(path, encoding="utf-8")) - 1
    print(f"{path}  ({rows} items)")

print()
print(open("data/benchmark/templates/ANNOTATION_INSTRUCTIONS.md", encoding="utf-8").read()[:1200])

## 7. Ingest completed annotations and compute agreement

**Run this only after the task files have actually been filled in.** Until then it correctly reports `NOT YET MEASURED` — that is the honest state, not a failure, and it must not be replaced with a placeholder number.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/run_annotation.py", "ingest", "--json"],
    capture_output=True, text=True,
)
print(result.stdout[-6000:])
if result.returncode != 0:
    print(result.stderr[-4000:])

## 8. Result

In [ ]:
from src.common.config import load_config
from src.common.paths import PathResolver

cfg = load_config()
resolver = PathResolver.from_config(cfg)
report = resolver.read_path("reports", "phase1_meer_annotation.md")
print(f"reading: {report}\n")
print(report.read_text())

---

### Saving the result as a Kaggle Dataset

1. Confirm `data/benchmark/` holds the pilot candidates, task files, and (once annotated) the ingested labels and agreement results.
2. **New Dataset** / **New Version**, e.g. `rbi-benchmark-pilot-v1`.
3. Download the task files locally so annotators can actually fill them in — they are the deliverable that unblocks the Fleiss' kappa.